In [1]:
import nibabel as nib        # For nifti files
import numpy as np           # For matrix math
import SimpleITK as sitk     # For N4 correction
import torchio as tio        # For deep learning
from dcm2niix import main as dcm2niix_run
from pathlib import Path
from dotenv import load_dotenv
import os
import ants
from nilearn.datasets import MNI152_FILE_PATH
import torch

#pipeline:
# 1. motion correction and n4 bias field correction

# 2. Skull stripping

# 3. spatial normalization

# 4. Intensity normalization

# 5. resizing (use placeholder values to signify dims)

# 6. Gaussian filters to smooth and denois data

# preserve as a 5d tensor for volumetric architecture



c:\Users\brand\Downloads\work\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

def run_volumetric_pipeline(nifti_path, template_path=MNI152_FILE_PATH): 
    # 1. Loading & Anatomical Orientation 
    subject = tio.Subject(t1w=tio.ScalarImage(str(nifti_path))) 
    to_ras = tio.ToCanonical() 
    subject = to_ras(subject) 
    
    sitk_img = subject.t1w.as_sitk() 
    if sitk_img.GetPixelID() != sitk.sitkFloat32: 
        sitk_img = sitk.Cast(sitk_img, sitk.sitkFloat32) 
        
    # 2. N4 Bias Field Correction & Otsu Masking (SimpleITK) 
    mask_img = sitk.OtsuThreshold(sitk_img, 0, 1, 200) 
    n4_corrector = sitk.N4BiasFieldCorrectionImageFilter() 
    n4_corrector.SetMaximumNumberOfIterations([50, 50, 50, 50]) 
    n4_corrector.SetConvergenceThreshold(1e-6) 
    corrected_sitk_img = n4_corrector.Execute(sitk_img, mask_img) 
    subject.t1w = tio.ScalarImage.from_sitk(corrected_sitk_img) 
    
    # 3. Skull Stripping (TorchIO Masking) 
    skull_stripper = tio.ForegroundMask(mask_name='brain_mask') 
    subject = skull_stripper(subject) 
    subject.t1w.data = subject.t1w.data * subject.brain_mask.data 

    # -------------------------------------------------------------------------
    # 4. FIXED: TRUE Spatial Normalization & Alignment using ANTs
    # -------------------------------------------------------------------------
    # Convert your processed image and the MNI template into ANTs images
    fixed_ants = ants.image_read(str(template_path))
    
    # Save a temporary file or convert to ANTs directly to realign the anatomy
    # ANTs will physically rotate and scale the brain to match MNI152 perfectly
    moving_ants = ants.from_numpy(
        subject.t1w.data.squeeze(0).numpy(), 
        spacing=subject.t1w.spacing, 
        origin=subject.t1w.origin, 
        direction=subject.t1w.direction.flatten()
    )
    
    # Run rigid + affine structural registration
    reg = ants.registration(fixed=fixed_ants, moving=moving_ants, type_of_transform='Affine')
    
    # Extract the truly aligned image and convert back to TorchIO
    registered_sitk = ants.to_libnet_image(reg['warpedmovout']) # internal sitk conversion
    subject.t1w = tio.ScalarImage.from_sitk(sitk.Cast(registered_sitk, sitk.sitkFloat32))

    # -------------------------------------------------------------------------
    # 5. Volumetric Standardizing (TorchIO Bounding Box) 
    # -------------------------------------------------------------------------
    target_shape = (128, 128, 128) 
    resizer = tio.CropOrPad(target_shape) 
    subject = resizer(subject) 
    
    # 6. Intensity Normalization (TorchIO Z-Score) 
    # Re-extract foreground mask on the newly warped brain coordinates
    subject = skull_stripper(subject)
    intensity_norm = tio.ZNormalization(masking_method='brain_mask') 
    subject = intensity_norm(subject) 
    
    # 8. Generation of the 5D Volumetric Network Tensor 
    final_4d_tensor = subject.t1w.data 
    final_5d_tensor = final_4d_tensor.unsqueeze(0) 
    
    return final_5d_tensor


In [14]:
# 1. Path to your top ADNI folder

load_dotenv()
folder_path = os.getenv('ADNI_FOLDER_PATH')
raw_root = Path(folder_path)

# 2. Path to where you want all converted .nii.gz files saved
output_root = Path("data/nifti_raw")

# Loop through every Participant folder inside cn cohort/ADNI
for subject_dir in raw_root.iterdir():
    if subject_dir.is_dir():
        print(f"Converting subject: {subject_dir.name}")
        
        # Create a matching subject directory in your output folder
        subj_output = output_root / subject_dir.name
        subj_output.mkdir(parents=True, exist_ok=True)
        
        # dcm2niix will automatically crawl down into MPRAGE -> Visits -> Cryptic ID -> DICOMs
        dcm2niix_run([
            "-z", "y",                 # Compress output to .nii.gz
            "-f", "%i_%p_%s",          # Filename style: ParticipantID_Protocol_Series
            "-o", str(subj_output),    # Destination folder for this participant
            str(subject_dir)           # Input directory (this participant's root folder)
        ])

print("All participant conversions complete!")

Converting subject: 009_S_0751


KeyboardInterrupt: 